In [ ]:
!pip install icrawler


In [ ]:
import os
import shutil
from icrawler.builtin import GoogleImageCrawler
from google.colab import files # Used for final download to local machine

SEARCH_NAME = "fancy wide jungle street view high resolution"
OUTPUT_DIR = "wide_junge"
NUM_IMAGES_TO_DOWNLOAD = 300
SIZE_FILTER = 'large'

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

google_crawler = GoogleImageCrawler(
    downloader_threads=4,
    storage={'root_dir': OUTPUT_DIR}
)

# 3. Configure and run the search
print(f"Searching for: '{SEARCH_NAME}'...")
print(f"Targeting {NUM_IMAGES_TO_DOWNLOAD} images (Size filter: '{SIZE_FILTER}')...")

google_crawler.crawl(
    keyword=SEARCH_NAME,
    max_num=NUM_IMAGES_TO_DOWNLOAD,
    min_size=(1280, 720), # Set a minimum resolution (e.g., HD)
    filters={'size': SIZE_FILTER},
    file_idx_offset=0
)

In [ ]:
len(os.listdir(r'/content/wide_junge'))


51

In [ ]:
shutil.rmtree(r'/content/output')


In [ ]:
import os
import random
import cv2
import numpy as np
from typing import List, Tuple, Optional, Dict, Any # <-- THIS IS THE KEY FIX

In [ ]:

BASE_DIR = 'output' 
SMALL_IMAGES_DIR = r'/content/drive/MyDrive/messbah/cropped' 
BACKGROUND_DIR = r"/content/wide_junge"
OUTPUT_IMAGES_DIR = os.path.join(BASE_DIR, 'augmented_images')
OUTPUT_LABELS_DIR = os.path.join(BASE_DIR, 'augmented_labels')

for d in [SMALL_IMAGES_DIR, BACKGROUND_DIR, OUTPUT_IMAGES_DIR, OUTPUT_LABELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Directories created/verified.")

# --- Helper Functions (Collision Detection & YOLO Labeling) ---

def get_yolo_label(class_id: int, x_start: int, y_start: int, w_obj: int, h_obj: int, w_bg: int, h_bg: int) -> List[float]:
    """Calculates normalized YOLO bounding box coordinates."""
    x_center_abs = x_start + (w_obj / 2)
    y_center_abs = y_start + (h_obj / 2)
    x_center_norm = x_center_abs / w_bg
    y_center_norm = y_center_abs / h_bg
    width_norm = w_obj / w_bg
    height_norm = h_obj / h_bg
    return [float(class_id), x_center_norm, y_center_norm, width_norm, height_norm]

def check_overlap(new_box: Tuple[int, int, int, int], existing_boxes: List[Tuple[int, int, int, int]], padding: int = 30) -> bool:
    """Checks if a new bounding box overlaps with any existing box (with padding)."""
    x1_new, y1_new, x2_new, y2_new = new_box

    x1_new_padded = x1_new - padding
    y1_new_padded = y1_new - padding
    x2_new_padded = x2_new + padding
    y2_new_padded = y2_new + padding

    for x1_old, y1_old, x2_old, y2_old in existing_boxes:

        if x2_old < x1_new_padded or x1_old > x2_new_padded or y2_old < y1_new_padded or y1_old > y2_new_padded:
            continue
        else:
            return True
    return False

def save_yolo_label_file(file_path: str, labels: List[List[float]]):
    """Saves all labels for a single image to a text file."""
    with open(file_path, 'w') as f:
        for label_data in labels:
            label_line = f"{int(label_data[0])} {label_data[1]:.6f} {label_data[2]:.6f} {label_data[3]:.6f} {label_data[4]:.6f}\n"
            f.write(label_line)


def paste_transparent_object(bg_img: np.ndarray, obj_path: str, existing_boxes: List[Tuple[int, int, int, int]], scale_range: Tuple[float, float] = (0.2, 0.6)) -> Optional[Dict[str, Any]]:
    """
    Loads transparent object, attempts non-overlapping placement, performs alpha blending.

    *** SCALE_RANGE: (0.2, 0.6) - Slightly increased scale to make objects a bit larger. ***
    """

    # 1. Load Object (must include alpha channel)
    obj_bgra = cv2.imread(obj_path, cv2.IMREAD_UNCHANGED)
    obj_filename = os.path.basename(obj_path)

    if obj_bgra is None:
        print(f"  DEBUG: Skip reason (Load): File '{obj_filename}' failed to load.")
        return None
    if obj_bgra.shape[2] < 4:
        print(f"  DEBUG: Skip reason (Alpha): Object '{obj_filename}' does not have an alpha channel.")
        return None

    # 2. Scaling & Initial Checks
    scale = random.uniform(scale_range[0], scale_range[1])
    h_obj_orig, w_obj_orig = obj_bgra.shape[:2]
    new_w = int(w_obj_orig * scale)
    new_h = int(h_obj_orig * scale)

    h_bg, w_bg = bg_img.shape[:2]

    if new_w >= w_bg * 0.9 or new_h >= h_bg * 0.9:
        print(f"  DEBUG: Skip reason (Size): Object '{obj_filename}' (scaled {new_w}x{new_h}) is too large for background ({w_bg}x{h_bg}).")
        return None
    if new_w < 15 or new_h < 15:
        return None
    
    MAX_ATTEMPTS = 50
    best_placement = None

    for attempt in range(MAX_ATTEMPTS):
        x_start = random.randint(0, w_bg - new_w)
        y_start = random.randint(0, h_bg - new_h)
        x_end = x_start + new_w
        y_end = y_start + new_h

        new_box = (x_start, y_start, x_end, y_end)

        # بررسی برخورد
        if not check_overlap(new_box, existing_boxes, padding=30):
            best_placement = new_box
            break

    if best_placement is None:
        return None

    x_start, y_start, x_end, y_end = best_placement

    obj_resized = cv2.resize(obj_bgra, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    b, g, r, alpha = cv2.split(obj_resized)
    obj_bgr_resized = cv2.merge((b, g, r))
    alpha_mask_normalized = alpha / 255.0

    roi = bg_img[y_start:y_end, x_start:x_end]
    inv_alpha_mask = 1.0 - alpha_mask_normalized

    alpha_3_channel = cv2.merge((alpha_mask_normalized, alpha_mask_normalized, alpha_mask_normalized))
    inv_alpha_3_channel = cv2.merge((inv_alpha_mask, inv_alpha_mask, inv_alpha_mask))

    foreground_part = cv2.multiply(alpha_3_channel, obj_bgr_resized.astype(float))
    background_part = cv2.multiply(inv_alpha_3_channel, roi.astype(float))
    blended_roi = cv2.add(foreground_part, background_part).astype(np.uint8)

    bg_img[y_start:y_end, x_start:x_end] = blended_roi

    class_id = 0
    yolo_data = get_yolo_label(class_id, x_start, y_start, new_w, new_h, w_bg, h_bg)

    return {
        "image": bg_img,
        "yolo_data": yolo_data,
        "bbox": best_placement
    }


def run_augmentation_pipeline(num_objects_per_bg: Tuple[int, int] = (4, 6), num_augmentations_per_bg: int = 5):
    """Runs the main augmentation loop with collision avoidance, creating multiple augmented images per background."""

    small_object_paths = [os.path.join(SMALL_IMAGES_DIR, f)
                          for f in os.listdir(SMALL_IMAGES_DIR) if f.endswith(('.png'))]
    background_paths = [os.path.join(BACKGROUND_DIR, f)
                        for f in os.listdir(BACKGROUND_DIR) if f.endswith(('.jpg', '.png'))]

    if not background_paths:
        print("\nFATAL ERROR: No background images found. Check BACKGROUND_DIR.")
        return
    if not small_object_paths:
        print("\nFATAL ERROR: No transparent object PNGs found. Check SMALL_IMAGES_DIR or ensure PNGs are 4-channel.")
        return

    print(f"\nFound {len(small_object_paths)} objects and {len(background_paths)} backgrounds.")

    # حلقه جدید برای ایجاد چندین خروجی از هر پس‌زمینه
    for bg_path in background_paths[:]:
        for i in range(num_augmentations_per_bg): 
            bg_filename = os.path.basename(bg_path)
            bg_name = os.path.splitext(bg_filename)[0]

            output_base_name = f"{bg_name}_aug_{i}"
            output_image_filename = f"{output_base_name}.{bg_filename.split('.')[-1]}" 
            output_label_filename = f"{output_base_name}.txt"

            print(f"\n--- Processing {bg_filename} for augmentation {i+1}/{num_augmentations_per_bg} ---")

            background_img = cv2.imread(bg_path)
            if background_img is None:
                print(f"Skipping {bg_filename}: Failed to load background image.")
                continue

            augmented_image = background_img.copy()
            new_labels: List[List[float]] = []
            placed_boxes: List[Tuple[int, int, int, int]] = []

            num_to_paste = random.randint(num_objects_per_bg[0], num_objects_per_bg[1])
            objects_to_paste = random.choices(small_object_paths, k=num_to_paste)

            print(f"Attempting to place {num_to_paste} non-overlapping object(s).")

            success_count = 0
            for obj_path in objects_to_paste:
                # استفاده از ضریب جدید (0.2, 0.6)
                result_data = paste_transparent_object(augmented_image, obj_path, placed_boxes, scale_range=(0.5, 0.99))

                if result_data is not None:
                    augmented_image = result_data["image"]
                    new_labels.append(result_data["yolo_data"])
                    placed_boxes.append(result_data["bbox"])
                    success_count += 1

            # --- Save Outputs ---
            if new_labels:
                img_output_path = os.path.join(OUTPUT_IMAGES_DIR, output_image_filename)
                cv2.imwrite(img_output_path, augmented_image)

                label_output_path = os.path.join(OUTPUT_LABELS_DIR, output_label_filename)
                save_yolo_label_file(label_output_path, new_labels)

                print(f"-> Successfully saved {success_count} object(s) and labels for {output_image_filename}")
            else:
                print(f"-> Skipped {output_image_filename}: No objects could be successfully placed.")

    print("\n--- Augmentation Complete! ---")
    print(f"Images saved to: {OUTPUT_IMAGES_DIR}")
    print(f"Labels saved to: {OUTPUT_LABELS_DIR}")


run_augmentation_pipeline(num_objects_per_bg=(5, 6), num_augmentations_per_bg=5)

In [ ]:
!zip -r /content/wide_jungle.zip output

In [ ]:
import cv2
import numpy as np

# ====================================================================
#  📍 CONFIGURATION: REPLACE THESE WITH YOUR ACTUAL FILE ADDRESSES
# ====================================================================
IMAGE_PATH = '/content/output/augmented_images/000014_aug_3.jpg' # Example: 'C:/Users/user/Desktop/data/photo.jpg' or '/home/user/data/photo.jpg'
LABEL_PATH = '/content/output/augmented_labels/000014_aug_3.txt' # Example: 'C:/Users/user/Desktop/data/photo.txt' or '/home/user/data/photo.txt'

# Define your class names (must match the order of class_ids in your label file)
# Example (if class_id 0 is 'person', 1 is 'car', etc.):
CLASS_NAMES = ['person']

# Simple color map for visualization
COLORS = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (0, 255, 255), (255, 0, 255)]
# ====================================================================

# --- 1. YOLO to Absolute Pixel Conversion Function ---

def yolo_to_abs(normalized_coords, width, height):
    """
    Converts YOLO normalized (center_x, center_y, w, h) to absolute (x_min, y_min, x_max, y_max)
    """
    center_x, center_y, w, h = normalized_coords

    w_abs = w * width
    h_abs = h * height
    x_center_abs = center_x * width
    y_center_abs = center_y * height

    x_min = int(x_center_abs - w_abs / 2)
    y_min = int(y_center_abs - h_abs / 2)
    x_max = int(x_center_abs + w_abs / 2)
    y_max = int(y_center_abs + h_abs / 2)

    return x_min, y_min, x_max, y_max

# --- 2. Main Plotting Logic ---

# 2.1 Load the image
img = cv2.imread(IMAGE_PATH)
if img is None:
    print(f"Error: Could not load image at {IMAGE_PATH}. Please check the path and file existence.")
    exit()

H, W, _ = img.shape

# 2.2 Read the label file
try:
    with open(LABEL_PATH, 'r') as f:
        labels_data = f.readlines()
except FileNotFoundError:
    print(f"Error: Could not find label file at {LABEL_PATH}. Please check the path.")
    exit()
except Exception as e:
    print(f"An error occurred while reading the label file: {e}")
    exit()

# 2.3 Process and Draw each bounding box
for line in labels_data:
    parts = line.strip().split()
    if len(parts) < 5:
        continue # Skip malformed or empty lines

    try:
        class_id = int(parts[0])
        # The next 4 are the normalized coordinates (center_x, center_y, w, h)
        normalized_coords = [float(p) for p in parts[1:5]]
    except ValueError:
        print(f"Skipping line with invalid numerical data: {line.strip()}")
        continue

    # Convert coordinates to absolute pixel values
    x_min, y_min, x_max, y_max = yolo_to_abs(normalized_coords, W, H)

    # Get class name and color
    class_name = CLASS_NAMES[class_id] if class_id < len(CLASS_NAMES) else f"Class {class_id}"
    color = COLORS[class_id % len(COLORS)]

    # --- Drawing ---

    # Draw Bounding Box (Rectangle)
    cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, 2)

    # Draw Label Text
    label = class_name
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_thickness = 2

    # Position the text slightly above the top-left corner
    text_size = cv2.getTextSize(label, font, font_scale, font_thickness)[0]
    text_x = x_min
    text_y = y_min - 10

    # Draw a solid background for the text
    cv2.rectangle(img,
                  (x_min, y_min - text_size[1] - 10),
                  (x_min + text_size[0] + 10, y_min),
                  color,
                  cv2.FILLED)

    # Put the text on the image
    cv2.putText(img, label,
                (text_x, text_y),
                font, font_scale,
                (255, 255, 255), # White text
                font_thickness, cv2.LINE_AA)

# --- 3. Display the Image ---

# cv2.imshow("YOLO Labels Plot", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
plt.imshow(img)

In [ ]:
import matplotlib.pyplot as plt
